# NumPy Matrix Ops Refresher — Demos + Immediate Practice

**Duration:** ~60 minutes  
**Audience:** Participants building LLMs with Python who need a fast but solid refresher on core NumPy operations used across ML.

**How to use this notebook**
- Each topic has a short *Demo* cell followed immediately by a *Practice* cell.  
- Run the demo, then complete the practice.  
- Topics build from basic array mechanics toward small ML‑style tasks.

**What you'll cover**
1. Arrays & shapes
2. Indexing, slicing, masking
3. Broadcasting
4. Vector & matrix multiplication (`dot`, `@`, `einsum`)
5. Norms and normalization / standardization
6. Axis‑wise reductions and softmax
7. One‑hot encoding & “embedding lookup”
8. Mini linear regression with the normal equation
9. (Bonus) Numerical stability: log‑sum‑exp

> Tip: Use `arr.shape`, `arr.ndim`, and `np.newaxis` frequently; they’re everywhere in deep learning code.

In [ ]:
# Setup
import numpy as np
np.set_printoptions(suppress=True, precision=4)
rng = np.random.default_rng(42)  # deterministic randomness
print("NumPy version:", np.__version__)
# Soft check: prints your result, or a hint if the TODO above isn't filled in yet.
# Keeps a fresh 'Run All' from throwing a traceback before you've written any code.
def check(label, fn):
    try:
        print(label, fn())
    except Exception as e:
        print(label, f"[fill in the TODO above] ({type(e).__name__})")

## 1) Arrays & shapes
Understand how vectors (1D), matrices (2D), and batched tensors (3D+) appear in ML code.

In [ ]:
# Demo: create arrays and inspect shapes
a = np.array([1., 2., 3.])         # vector
B = np.array([[1., 2., 3.],        # matrix (2x3)
              [4., 5., 6.]])
C = rng.normal(size=(4, 3, 2))     # batch of matrices (batch=4)

print("a.shape:", a.shape, "| a.ndim:", a.ndim)
print("B.shape:", B.shape, "| B.ndim:", B.ndim)
print("C.shape:", C.shape, "| C.ndim:", C.ndim)

# Reshape and ravel: common in preparing features
x = rng.normal(size=12)
X = x.reshape(3, 4)  # 3 rows, 4 cols
print("X:\n", X)
print("X.ravel():", X.ravel())

In [ ]:
# Practice: Given a flat vector of length 24, reshape to (batch=3, time=4, features=2).
v = rng.normal(size=24)
# TODO: set T to the correctly shaped view of v
T = ...  # shape should be (3, 4, 2)
check("T.shape should be (3, 4, 2) ->", lambda: T.shape)

## 2) Indexing, slicing, masking
Slicing along axes and boolean masks are essential for data filtering and minibatching.

In [ ]:
# Demo: slicing rows/cols and boolean masks
X = rng.normal(loc=0.0, scale=1.0, size=(5, 4))  # 5 samples, 4 features
print("X:\n", X)

first_two_rows = X[:2]
last_column = X[:, -1]
big_mask = X[:, 0] > 0.0  # samples where feature0 > 0
X_big = X[big_mask]

print("first_two_rows shape:", first_two_rows.shape)
print("last_column shape:", last_column.shape)
print("mask:", big_mask)
print("X_big shape:", X_big.shape)

In [ ]:
# Practice: From X, select rows where the mean across features > 0,
# then select only columns 1 and 3 (zero-based).
X = rng.normal(size=(8, 5))
# TODO: build a mask over rows
row_mask = ...
# TODO: slice the filtered matrix to get columns 1 and 3 only
X_sel = ...

print("Row mask:", row_mask)
check("X_sel shape should be (num_selected, 2) ->", lambda: X_sel.shape)

## 3) Broadcasting
Align shapes without explicit loops. Core to normalization and batched math.

In [ ]:
# Demo: feature-wise centering using broadcasting
X = rng.normal(size=(6, 3))  # 6 samples, 3 features
mu = X.mean(axis=0)          # (3,)
centered = X - mu            # (6,3) - (3,) -> (6,3)
print("mu:", mu)
print("centered mean (approx 0):", centered.mean(axis=0))

In [ ]:
# Practice: Scale each feature by its standard deviation using broadcasting.
X = rng.normal(size=(10, 4))
mu = X.mean(axis=0)
sigma = X.std(axis=0, ddof=0)  # population std

# TODO: standardize X to Z with zero mean and unit variance per feature
Z = ...

check("Z mean (approx 0):", lambda: Z.mean(axis=0))
check("Z std  (approx 1):", lambda: Z.std(axis=0))

## 4) Vector & matrix multiplication (`dot`, `@`, `einsum`)
Compute linear combinations and batched projections.

In [ ]:
# Demo: linear model y = Xw + b
X = rng.normal(size=(5, 3))
w = rng.normal(size=(3,))   # weights
b = 0.5
y = X @ w + b               # same as np.dot(X, w) + b
print("y:", y)

# einsum: explicit dimension mapping (useful for clarity/perf)
y2 = np.einsum('ij,j->i', X, w) + b
print("y2 (should match y):", y2)

In [ ]:
# Practice: Given batch of 7 vectors with 4 features, project to 2-D using W (4x2),
# then add bias vector b (2,).
X = rng.normal(size=(7, 4))
W = rng.normal(size=(4, 2))
b = rng.normal(size=(2,))

# TODO: compute Y of shape (7, 2)
Y = ...
check("Y.shape should be (7, 2) ->", lambda: Y.shape)

## 5) Norms and normalization / standardization
L2 norms and unit‑length feature vectors are common in similarity tasks.

In [ ]:
# Demo: L2-normalize rows (common before cosine similarity)
X = rng.normal(size=(4, 5))
l2 = np.linalg.norm(X, ord=2, axis=1, keepdims=True)  # (4,1)
X_unit = X / (l2 + 1e-12)  # avoid divide-by-zero
row_norms = np.linalg.norm(X_unit, axis=1)
print("Row norms after normalization (approx 1):", row_norms)

In [ ]:
# Practice: Standardize features (columns) to zero mean and unit variance.
X = rng.normal(loc=3.0, scale=2.0, size=(12, 6))
# TODO: compute column means and stds, then standardize
mu = ...
sd = ...
Z = ...

check("mean ~0:", lambda: Z.mean(axis=0))
check("std  ~1:", lambda: Z.std(axis=0))

## 6) Axis‑wise reductions and softmax
Summations/means across axes and a numerically stable softmax.

In [ ]:
# Demo: stable softmax over classes (axis=1)
def softmax(logits, axis=-1):
    z = logits - logits.max(axis=axis, keepdims=True)  # stability shift
    exp = np.exp(z)
    return exp / exp.sum(axis=axis, keepdims=True)

logits = rng.normal(size=(3, 5))  # 3 samples, 5 classes
probs = softmax(logits, axis=1)
print("Row sums (should be 1):", probs.sum(axis=1))
print("Argmax classes:", probs.argmax(axis=1))

In [ ]:
# Practice: Implement a column-wise softmax (axis=0) for a (6, 4) matrix of logits.
L = rng.normal(size=(6, 4))
# TODO: implement col_softmax so each column sums to 1
def col_softmax(A):
    ...
P = col_softmax(L)
check("Col sums (should be ~1):", lambda: P.sum(axis=0))

## 7) One‑hot encoding & “embedding lookup”
Map integer token IDs to rows of an embedding matrix; core to NLP & LLMs.

In [ ]:
# Demo: one-hot and lookup
vocab_size = 8
embed_dim = 4
token_ids = np.array([3, 1, 0, 6])   # length 4 sequence
E = rng.normal(size=(vocab_size, embed_dim))

# One-hot via eye:
one_hot = np.eye(vocab_size)[token_ids]   # (4, vocab_size)
# Equivalent "embedding lookup" via gather:
embeds = E[token_ids]                     # (4, embed_dim)

print("one_hot shape:", one_hot.shape)
print("embeds shape:", embeds.shape)
# Check equivalence: one_hot @ E == embeds
print("Equivalent lookup:", np.allclose(one_hot @ E, embeds))

In [ ]:
# Practice: Given a batch of sequences of token ids, produce average-pooled embeddings.
batch_token_ids = np.array([[0, 1, 1, 2],
                            [3, 3, 4, 5],
                            [6, 7, 7, 7]])
E = rng.normal(size=(10, 6))  # vocab=10, dim=6

# TODO: compute (batch, dim) pooled embeddings by averaging per sequence
# Hint: E[batch_token_ids] -> (batch, seq_len, dim)
avg_embeds = ...
check("avg_embeds shape should be (3, 6) ->", lambda: avg_embeds.shape)

## 8) Mini linear regression with the normal equation
Fit \(w, b\) by closed form. Useful to connect linear algebra with ML loss minimization.

In [ ]:
# Demo: generate synthetic linear data and solve (X^T X)^{-1} X^T y
n, d = 100, 3
X = rng.normal(size=(n, d))
true_w = rng.normal(size=d)
true_b = -0.7
noise = rng.normal(scale=0.1, size=n)
y = X @ true_w + true_b + noise

# Add bias column of 1s
X_aug = np.c_[X, np.ones(n)]
theta_hat = np.linalg.pinv(X_aug) @ y   # (d+1,)
w_hat, b_hat = theta_hat[:-1], theta_hat[-1]

mse = np.mean((X @ w_hat + b_hat - y)**2)
print("true_w:", true_w, "true_b:", true_b)
print("w_hat :", w_hat,  "b_hat :", b_hat)
print("train MSE:", mse)

In [ ]:
# Practice: Using the fitted (w_hat, b_hat), compute predictions on new samples
# and report MSE against a fresh target generated by the true params.
X_new = rng.normal(size=(50, 3))
noise = rng.normal(scale=0.1, size=50)
y_true = X_new @ true_w + true_b + noise

# TODO: compute y_pred and mse_new
y_pred = ...
mse_new = ...
print("MSE on new data (should be close to noise variance 0.01):", mse_new)

## 9) (Bonus) Numerical stability — log‑sum‑exp
Avoid overflow/underflow when turning logits into probabilities.

In [ ]:
# Demo + Practice: implement logsumexp that is stable
def logsumexp(x, axis=None, keepdims=False):
    m = x.max(axis=axis, keepdims=True)
    y = np.log(np.exp(x - m).sum(axis=axis, keepdims=True)) + m
    if not keepdims:
        y = np.squeeze(y, axis=axis)
    return y

big = np.array([1000.0, 1001.0, 1002.0])
naive = np.log(np.exp(big).sum())
stable = logsumexp(big)
print("Naive (inf expected):", naive)
print("Stable:", stable)

---

## Wrap‑up
You reviewed shapes, slicing/masking, broadcasting, matrix multiplication, normalization, reductions/softmax, one‑hot/embeddings, and a tiny linear regression—all pillars for reading & writing ML/LLM NumPy code.